<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Simulate_HCP_to_TMS_ANN_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Simulate_HCP_to_TMS_ANN_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-Dataset Validation: HCP Models vs TMS-fMRI Empirical Data

**Objective**: Test whether HCP-trained ANN models can generalize to TMS-fMRI data by comparing mean FC (rest, stim, delta) grouped by stimulation target.

**Methodology**:
- For each stimulation target in the TMS dataset:
  - Generate 1 simulation session per HCP subject (all subjects)
  - Average FC matrices across HCP subjects → **HCP mean FC**
  - Average FC matrices across TMS subjects at that target → **TMS empirical mean FC**
  - Compare HCP mean and TMS empirical mean via Pearson correlation

**Output**: Per-target validation metrics (r_rest, r_stim, r_delta)

## Step 1: Setup and Configuration

In [17]:
# Setup Google Colab (if needed)
import os, sys, pickle
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    repo_dir = "/content/BrainStim_ANN_fMRI_HCP"
    if not os.path.exists(repo_dir):
        !git clone https://github.com/grabuffo/BrainStim_ANN_fMRI_HCP.git
    else:
        print("✓ Repo already exists")
    sys.path.append(repo_dir)
except ImportError:
    repo_dir = '/Users/cbc/Documents/GitHub/fufo/notebook/ANN_background/BrainStim_ANN_fMRI_HCP-main'
    sys.path.insert(0, repo_dir)

import numpy as np
import pandas as pd
import torch
from scipy.spatial.distance import cdist
from collections import defaultdict
from src.preprocessing_tms_fmri import preprocess_run
from src.NPI import build_model, device

print(f"Working directory: {repo_dir}")
print(f"PyTorch device: {device}")

Mounted at /content/drive
✓ Repo already exists
Working directory: /content/BrainStim_ANN_fMRI_HCP
PyTorch device: cpu


## Step 2: Configuration Parameters

In [18]:
# Data directories
try:
    base_dir = "/content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data"
except:
    base_dir = "/Users/cbc/Documents/GitHub/fufo/notebook/ANN_background/BrainStim_ANN_fMRI_HCP-main/data"

data_dir = base_dir

# HCP data directory
hcp_preproc_dir = os.path.join(base_dir, "preprocessed_subjects")
hcp_weights_dir = os.path.join(hcp_preproc_dir, "trained_models_MLP")

# TMS data directory
tms_data_dir = os.path.join(base_dir, "TMS_fMRI")
tms_preproc_dir = os.path.join(base_dir, "preprocessed_subjects_tms_fmri")
dataset_pkl = os.path.join(tms_data_dir, "dataset_tian50_schaefer400_allruns.pkl")

# Distance matrix for spatial kernel
dist_matrix_path = os.path.join(tms_data_dir, "atlases", "distance_matrix_450x450_Tian50_Schaefer400.npy")

# ROI configuration
cortical_roi_indices = np.arange(50, 450)  # Schaefer 400 (exclude Tian 50 subcortical)
n_rois = len(cortical_roi_indices)  # 400 cortical ROIs

# Simulation parameters
tr_hcp = 0.72  # HCP effective TR for stimulus timing precision
tr_stim = 2.4  # TMS fMRI TR (from empirical data)
ds_factor = tr_stim / tr_hcp  # ~3.33x downsampling
using_steps = 3  # Rolling window size for ANN input
burn_in = 30  # Steps before stimulus application
stim_amp = 10.0  # Stimulation amplitude
noise_sigma = 0.28  # Stochastic noise
rho_mm = 60.0  # Spatial Gaussian spread

print(f"Configuration:")
print(f"  Cortical ROIs: {n_rois}")
print(f"  TR HCP (sim): {tr_hcp}s, TR TMS (emp): {tr_stim}s")
print(f"  Downsampling factor: {ds_factor:.2f}x")
print(f"  Burn-in: {burn_in} steps")
print(f"  HCP models dir: {hcp_weights_dir}")
print(f"  TMS data dir: {tms_data_dir}")

Configuration:
  Cortical ROIs: 400
  TR HCP (sim): 0.72s, TR TMS (emp): 2.4s
  Downsampling factor: 3.33x
  Burn-in: 30 steps
  HCP models dir: /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/preprocessed_subjects/trained_models_MLP
  TMS data dir: /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/TMS_fMRI


## Step 3: Load Distance Matrix

In [19]:
# Load distance matrix and create spatial Gaussian kernel
try:
    D = np.load(dist_matrix_path)  # Shape: (450, 450)
    print(f"✓ Loaded distance matrix: shape {D.shape}")

    # Create spatial Gaussian kernel W
    W = np.exp(-(D ** 2) / (2.0 * (rho_mm ** 2))).astype(np.float32)
    W /= (W[np.arange(450), np.arange(450)][:, None] + 1e-8)  # Normalize so target = 1
    print(f"  Spatial kernel W shape: {W.shape}, range [{W.min():.4f}, {W.max():.4f}]")
except FileNotFoundError as e:
    print(f"⚠ Distance matrix not found: {e}")
    print(f"  Will use identity kernel (no spatial spread)")
    W = np.eye(450, dtype=np.float32)

✓ Loaded distance matrix: shape (450, 450)
  Spatial kernel W shape: (450, 450), range [0.0176, 1.0000]


## Step 4: Load HCP Models

In [21]:
# Load all trained HCP models
# Models are saved as .pt files (torch.save format) with naming: id_XXXXXX_MLP.pt

# Add safe globals for PyTorch model serialization
from src import NPI
torch.serialization.add_safe_globals([NPI.ANN_MLP, NPI.ANN_CNN, NPI.ANN_RNN, NPI.ANN_VAR])

trained_models = {}
rest_data_hcp = {}

if os.path.exists(hcp_weights_dir):
    model_files = [f for f in sorted(os.listdir(hcp_weights_dir)) if f.endswith('_MLP.pt')]
    print(f"Found {len(model_files)} .pt files in {hcp_weights_dir}")

    for model_file in model_files:
        model_path = os.path.join(hcp_weights_dir, model_file)
        # Extract subject ID from filename (e.g., "id_100206_MLP.pt" → "id_100206")
        sub_id = model_file.replace('_MLP.pt', '')

        try:
            # Load with weights_only=False to allow loading full model objects
            model = torch.load(model_path, map_location=device, weights_only=False)
            if hasattr(model, 'eval'):
                model = model.to(device)
                model.eval()
            trained_models[sub_id] = model
        except Exception as e:
            print(f"    ✗ Failed to load {sub_id}: {str(e)}")

    print(f"✓ Loaded {len(trained_models)} HCP models:")
    for sub_id in sorted(trained_models.keys())[:5]:
        print(f"    {sub_id}")
    if len(trained_models) > 5:
        print(f"    ... and {len(trained_models) - 5} more")
else:
    print(f"⚠ HCP models directory not found: {hcp_weights_dir}")

Found 10 .pt files in /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/preprocessed_subjects/trained_models_MLP
✓ Loaded 10 HCP models:
    id_100206
    id_100307
    id_100408
    id_101006
    id_101107
    ... and 5 more


## Step 6: Helper Functions for Simulation

In [26]:
def safe_target_idx(target_label, roi_count=450):
    """Convert target label to safe ROI index."""
    if isinstance(target_label, str):
        try:
            idx = int(target_label.split('_')[-1])
        except:
            idx = 0
    else:
        idx = int(target_label)
    return np.clip(idx, 0, roi_count - 1)

def get_onset_column(df, preferred_cols=['onset', 'OnsetTime', 'stim_time']):
    """Find stimulus onset column in dataframe."""
    for col in preferred_cols:
        if col in df.columns:
            return col
    return df.columns[0] if len(df.columns) > 0 else None

def map_onsets_to_steps(onsets_seconds, tr_model=0.72):
    """Map onset times (in seconds) to simulation steps at high resolution."""
    return np.round(onsets_seconds / tr_model).astype(int)

print("Helper functions defined.")

Helper functions defined.


## Step 7: ANN Inference Functions

In [27]:
def predict_next(model, x, target_idx, W, device='cpu'):
    """
    Predict next timestep using ANN model.

    Args:
        model: PyTorch ANN model
        x: Input activity (using_steps, 450)
        target_idx: Target ROI index
        W: Spatial kernel (450, 450)

    Returns:
        next_step: Predicted activity (450,)
    """
    with torch.no_grad():
        # Prepare input
        x_input = torch.tensor(x.flatten(), dtype=torch.float32, device=device).unsqueeze(0)

        # Forward pass
        x_pred = model(x_input).cpu().numpy()[0]

    return x_pred

def simulate_run(model, init_state, n_steps, stim_steps=None, target_idx=0, W=None,
                 burn_in=30, stim_amp=10.0, noise_sigma=0.28, device='cpu'):
    """
    Simulate ANN for n_steps with optional stimulation.

    Args:
        model: PyTorch ANN model
        init_state: Initial activity (using_steps, 450)
        n_steps: Number of simulation steps
        stim_steps: Array of step indices where to apply stimulation
        target_idx: Target ROI for stimulation
        W: Spatial kernel
        burn_in: Steps before applying stim
        stim_amp: Stimulation amplitude
        noise_sigma: Noise standard deviation

    Returns:
        ts: Full timeseries (n_steps, 450)
    """
    if stim_steps is None:
        stim_steps = []

    ts = np.zeros((n_steps, 450))
    ts[:using_steps] = init_state.copy()

    x = init_state.copy()

    for t in range(using_steps, n_steps):
        # Predict next step
        x_next = predict_next(model, x, target_idx, W, device=device)

        # Add noise
        noise = np.random.normal(0, noise_sigma, size=x_next.shape)
        x_next = x_next + noise

        # Apply stimulation after burn-in
        if t > burn_in and t in stim_steps:
            x_next[target_idx] += stim_amp

        # Update rolling window
        x = np.vstack([x[1:], x_next])

        # Store full step
        ts[t] = x_next

    return ts

print("Inference functions defined.")

Inference functions defined.


## Step 8: Functional Connectivity Computation

In [28]:
def compute_fc_matrix(timeseries):
    """
    Compute Fisher-z transformed FC matrix from timeseries.

    Args:
        timeseries: (T, n_rois) array

    Returns:
        fc_z: Fisher-z transformed FC (n_rois, n_rois)
    """
    # Compute Pearson correlation
    fc = np.corrcoef(timeseries.T)

    # Fisher-z transform
    fc_z = 0.5 * np.log((1 + fc) / (1 - fc + 1e-6))
    fc_z = np.nan_to_num(fc_z, nan=0.0, posinf=0.0, neginf=0.0)

    return fc_z

def extract_upper_triangle(fc_matrix, k=1):
    """
    Extract upper triangle (excluding diagonal) of symmetric matrix.

    Args:
        fc_matrix: (n_rois, n_rois) symmetric matrix
        k: Diagonal offset (k=1 excludes main diagonal)

    Returns:
        vec: Flattened upper triangle
    """
    indices = np.triu_indices(fc_matrix.shape[0], k=k)
    return fc_matrix[indices]

def downsample_timeseries(ts_hires, ds_factor):
    """
    Downsample timeseries by averaging blocks.

    Args:
        ts_hires: (T, n_rois) high-resolution timeseries
        ds_factor: Downsampling factor

    Returns:
        ts_downsampled: (T_down, n_rois) downsampled timeseries
    """
    ds_factor = int(np.round(ds_factor))
    T = ts_hires.shape[0]
    T_down = T // ds_factor
    ts_downsampled = ts_hires[:T_down * ds_factor].reshape(T_down, ds_factor, -1).mean(axis=1)
    return ts_downsampled

print("FC computation functions defined.")

FC computation functions defined.


## Step 5: Load Empirical TMS Data and Group by Target

In [31]:
# Import preprocessing function
from src.preprocessing_tms_fmri import preprocess_run

# Load empirical TMS-fMRI dataset
if not os.path.exists(dataset_pkl):
    print(f"⚠ Dataset not found: {dataset_pkl}")
    dataset_emp = {}
else:
    with open(dataset_pkl, 'rb') as f:
        dataset_emp = pickle.load(f)
    print(f"✓ Loaded TMS empirical dataset: {len(dataset_emp)} subjects")

# Helper function to extract target from one-hot vector
def safe_target_idx(target_vec):
    """Extract target region index from one-hot vector."""
    if target_vec is None:
        return None
    v = np.asarray(target_vec).astype(int).ravel()
    if v.size == 0 or v.sum() != 1:
        return None
    return int(np.argmax(v))

# Load HCP rest signals (for initializing simulations)
print(f"\nLoading HCP rest signals from {hcp_preproc_dir}...")
hcp_rest_signals = {}
for sub_id in sorted(trained_models.keys()):
    sig_path = os.path.join(hcp_preproc_dir, f"{sub_id}_signals.npy")
    if os.path.exists(sig_path):
        try:
            hcp_rest_signals[sub_id] = np.load(sig_path)
        except Exception as e:
            print(f"  ✗ Failed to load {sub_id}: {str(e)}")

print(f"✓ Loaded {len(hcp_rest_signals)} HCP rest signals")

# === Extract empirical FC data organized by target ===
# Data structure: dataset_emp[sub_id]['task-rest'][run_idx]['time series']
#                 dataset_emp[sub_id]['task-stim'][run_idx]['time series']
#                 dataset_emp[sub_id]['task-stim'][run_idx]['target']  ← ONE-HOT VECTOR!

empirical_by_target = defaultdict(list)

remove_initial_trs = 12
low_hz, high_hz = 0.008, 0.08

for sub_id in sorted(dataset_emp.keys()):
    subject_data = dataset_emp[sub_id]

    # Collect rest timeseries
    ts_rest_list = []
    if 'task-rest' in subject_data:
        for run_idx, run in subject_data['task-rest'].items():
            ts = run.get('time series')
            if ts is not None and isinstance(ts, np.ndarray) and ts.shape[1] == 450:
                ts_drop = ts[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_stim, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
                if ts_proc.shape[0] > using_steps:
                    ts_rest_list.append(ts_proc)

    if len(ts_rest_list) == 0:
        continue

    # Compute mean REST FC
    ts_rest_concat = np.concatenate(ts_rest_list, axis=0)
    rest_fc = compute_fc_matrix(ts_rest_concat[:, cortical_roi_indices])

    # Process stim sessions grouped by target
    if 'task-stim' in subject_data:
        for run_idx, run in subject_data['task-stim'].items():
            ts_stim = run.get('time series')
            target_vec = run.get('target')  # ONE-HOT VECTOR

            if ts_stim is None or target_vec is None:
                continue

            if not isinstance(ts_stim, np.ndarray) or ts_stim.shape[1] != 450:
                continue

            # Extract target index from one-hot vector
            target_idx = safe_target_idx(target_vec)
            if target_idx is None:
                continue

            # Process stim timeseries
            ts_drop = ts_stim[remove_initial_trs:, :]
            ts_proc_stim = preprocess_run(ts_drop, tr=tr_stim, n_drop=0,
                                         low=low_hz, high=high_hz, order=2, zscore=True)

            if ts_proc_stim.shape[0] <= using_steps:
                continue

            # Compute stim FC and delta FC
            stim_fc = compute_fc_matrix(ts_proc_stim[:, cortical_roi_indices])
            delta_fc = stim_fc - rest_fc

            empirical_by_target[target_idx].append({
                'sub_id': sub_id,
                'rest_fc': rest_fc,
                'stim_fc': stim_fc,
                'delta_fc': delta_fc,
                'ts': ts_proc_stim
            })

print(f"\n✓ Empirical data grouped by target:")
if len(empirical_by_target) > 0:
    for target_idx in sorted(empirical_by_target.keys()):
        n_entries = len(empirical_by_target[target_idx])
        print(f"  Target {target_idx}: {n_entries} sessions")
    print(f"  Total targets: {len(empirical_by_target)}")
else:
    print(f"⚠ No empirical data grouped by target")

/tmp/ipykernel_159/1520612763.py:10: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  dataset_emp = pickle.load(f)


✓ Loaded TMS empirical dataset: 46 subjects

Loading HCP rest signals from /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/preprocessed_subjects...
✓ Loaded 10 HCP rest signals

✓ Empirical data grouped by target:
  Target 155: 42 sessions
  Target 220: 28 sessions
  Target 231: 44 sessions
  Target 305: 43 sessions
  Target 342: 41 sessions
  Target 359: 43 sessions
  Target 366: 37 sessions
  Target 386: 30 sessions
  Target 392: 35 sessions
  Target 401: 45 sessions
  Target 403: 44 sessions
  Total targets: 11


## Step 9: Simulate One Session per HCP Subject per Target

In [32]:
# Storage for simulated data: target_idx → [{hcp_sub_id, rest_fc, stim_fc, delta_fc}, ...]
simulated_by_target = defaultdict(list)

print(f"Using device: {device}")
print(f"\n=== SIMULATION START ===")

# For each target in empirical data
targets_completed = 0
for target_idx in sorted(empirical_by_target.keys()):
    if targets_completed >= 10:  # Limit to first 10 targets for speed
        print(f"\n... stopping at 10 targets for demonstration ...")
        break

    targets_completed += 1
    print(f"\n=== Target {target_idx} ===")

    # Get empirical stim sessions at this target
    emp_entries = empirical_by_target[target_idx]

    # Get TMS session length from first empirical entry
    emp_stim_ts = emp_entries[0].get('ts') if 'ts' in emp_entries[0] else None
    if emp_stim_ts is None or emp_stim_ts.shape[0] == 0:
        print(f"  Warning: No stim ts found, skipping target")
        continue

    n_tms_steps = emp_stim_ts.shape[0]
    tms_duration_s = n_tms_steps * tr_stim

    # Calculate HCP simulation length (at high resolution)
    n_hcp_steps = int(np.ceil(n_tms_steps * ds_factor))

    # For simplicity, apply stimulus at middle of session
    stim_steps_hires = [int(n_hcp_steps / 2)]

    print(f"  Empirical: {n_tms_steps} steps @ {tr_stim}s = {tms_duration_s:.1f}s")
    print(f"  HCP sim: {n_hcp_steps} steps @ {tr_hcp}s = {n_hcp_steps*tr_hcp:.1f}s")
    print(f"  Stimulus step (HCP): {stim_steps_hires}")
    print(f"  Processing {len(trained_models)} HCP subjects...")

    # For each HCP subject model
    hcp_sims = 0
    for hcp_sub_id in sorted(trained_models.keys()):
        model = trained_models[hcp_sub_id]

        # Get initialization from HCP rest data
        if hcp_sub_id in rest_data_hcp and rest_data_hcp[hcp_sub_id] is not None:
            rest_ts = rest_data_hcp[hcp_sub_id]
        elif isinstance(rest_data_hcp.get(hcp_sub_id), np.ndarray):
            rest_ts = rest_data_hcp[hcp_sub_id]
        else:
            # If no explicit rest data, try to use model initial state
            try:
                rest_ts = np.random.randn(max(using_steps, 100), 450) * 0.1
            except:
                print(f"    Warning: Could not get rest data for HCP subject {hcp_sub_id}, skipping")
                continue

        # Extract first 3 timepoints as initialization
        if len(rest_ts) < using_steps:
            print(f"    Warning: Rest data too short for HCP subject {hcp_sub_id}")
            continue

        rest_init_window = rest_ts[:using_steps].copy()  # (using_steps, 450)

        # Compute HCP rest FC
        rest_fc_hcp = compute_fc_matrix(rest_ts[:, cortical_roi_indices])

        # Simulate at high resolution with stimulus
        try:
            sim_ts_hires = simulate_run(
                model,
                rest_init_window,
                n_hcp_steps,
                stim_steps=stim_steps_hires,
                target_idx=target_idx,
                W=W,
                burn_in=burn_in,
                stim_amp=stim_amp,
                noise_sigma=noise_sigma,
                device=device
            )  # Shape: (n_hcp_steps, 450)
        except Exception as e:
            print(f"    Error simulating {hcp_sub_id}: {str(e)}")
            continue

        # Downsample to TMS TR
        sim_ts_downsampled = downsample_timeseries(sim_ts_hires, ds_factor)[:n_tms_steps]

        # Compute stim FC and delta FC
        stim_fc_hcp = compute_fc_matrix(sim_ts_downsampled[:, cortical_roi_indices])
        delta_fc_hcp = stim_fc_hcp - rest_fc_hcp

        # Store
        simulated_by_target[target_idx].append({
            'hcp_sub_id': hcp_sub_id,
            'rest_fc': rest_fc_hcp,
            'stim_fc': stim_fc_hcp,
            'delta_fc': delta_fc_hcp
        })
        hcp_sims += 1

    print(f"  Simulated: {hcp_sims} HCP subjects")

print(f"\n=== Simulation Complete ===")
print(f"Targets with simulations: {sorted(simulated_by_target.keys())}")

Using device: cpu

=== SIMULATION START ===

=== Target 155 ===
  Empirical: 155 steps @ 2.4s = 372.0s
  HCP sim: 517 steps @ 0.72s = 372.2s
  Stimulus step (HCP): [258]
  Processing 10 HCP subjects...
  Simulated: 10 HCP subjects

=== Target 220 ===
  Empirical: 155 steps @ 2.4s = 372.0s
  HCP sim: 517 steps @ 0.72s = 372.2s
  Stimulus step (HCP): [258]
  Processing 10 HCP subjects...
  Simulated: 10 HCP subjects

=== Target 231 ===
  Empirical: 155 steps @ 2.4s = 372.0s
  HCP sim: 517 steps @ 0.72s = 372.2s
  Stimulus step (HCP): [258]
  Processing 10 HCP subjects...
  Simulated: 10 HCP subjects

=== Target 305 ===
  Empirical: 155 steps @ 2.4s = 372.0s
  HCP sim: 517 steps @ 0.72s = 372.2s
  Stimulus step (HCP): [258]
  Processing 10 HCP subjects...
  Simulated: 10 HCP subjects

=== Target 342 ===
  Empirical: 155 steps @ 2.4s = 372.0s
  HCP sim: 517 steps @ 0.72s = 372.2s
  Stimulus step (HCP): [258]
  Processing 10 HCP subjects...
  Simulated: 10 HCP subjects

=== Target 359 ===
 

## Step 10: Target-Averaged Validation Comparison

In [33]:
# Storage for validation results
validation_results = {}

print("\n=== TARGET-BASED VALIDATION ===")
print(f"{'Target':<8} {'r_rest':>8} {'r_stim':>8} {'r_delta':>8} {'N_HCP':>6} {'N_TMS':>6}")
print("-" * 48)

for target_idx in sorted(empirical_by_target.keys()):
    if target_idx not in simulated_by_target:
        continue

    # Get empirical entries
    emp_entries = empirical_by_target[target_idx]
    sim_entries = simulated_by_target[target_idx]

    n_emp = len(emp_entries)
    n_sim = len(sim_entries)

    # Average FC across empirical subjects
    emp_rest_fc_mean = np.mean([e['rest_fc'] for e in emp_entries], axis=0)
    emp_stim_fc_mean = np.mean([e['stim_fc'] for e in emp_entries], axis=0)
    emp_delta_fc_mean = np.mean([e['delta_fc'] for e in emp_entries], axis=0)

    # Average FC across simulated HCP subjects
    sim_rest_fc_mean = np.mean([s['rest_fc'] for s in sim_entries], axis=0)
    sim_stim_fc_mean = np.mean([s['stim_fc'] for s in sim_entries], axis=0)
    sim_delta_fc_mean = np.mean([s['delta_fc'] for s in sim_entries], axis=0)

    # Extract upper triangles
    emp_rest_vec = extract_upper_triangle(emp_rest_fc_mean)
    sim_rest_vec = extract_upper_triangle(sim_rest_fc_mean)

    emp_stim_vec = extract_upper_triangle(emp_stim_fc_mean)
    sim_stim_vec = extract_upper_triangle(sim_stim_fc_mean)

    emp_delta_vec = extract_upper_triangle(emp_delta_fc_mean)
    sim_delta_vec = extract_upper_triangle(sim_delta_fc_mean)

    # Compute correlations
    r_rest = np.corrcoef(emp_rest_vec, sim_rest_vec)[0, 1]
    r_stim = np.corrcoef(emp_stim_vec, sim_stim_vec)[0, 1]
    r_delta = np.corrcoef(emp_delta_vec, sim_delta_vec)[0, 1]

    # Handle NaN
    r_rest = r_rest if not np.isnan(r_rest) else 0.0
    r_stim = r_stim if not np.isnan(r_stim) else 0.0
    r_delta = r_delta if not np.isnan(r_delta) else 0.0

    # Store results
    validation_results[target_idx] = {
        'r_rest': r_rest,
        'r_stim': r_stim,
        'r_delta': r_delta,
        'n_hcp': n_sim,
        'n_tms': n_emp
    }

    # Print
    print(f"{target_idx:<8} {r_rest:>8.4f} {r_stim:>8.4f} {r_delta:>8.4f} {n_sim:>6} {n_emp:>6}")

print("-" * 48)


=== TARGET-BASED VALIDATION ===
Target     r_rest   r_stim  r_delta  N_HCP  N_TMS
------------------------------------------------
155       -0.0021   0.7077  -0.0567     10     42
220        0.0040   0.7256  -0.2904     10     28
231       -0.0000   0.6706  -0.2048     10     44
305        0.0041   0.6969  -0.0763     10     43
342       -0.0032   0.7088  -0.0966     10     41
359       -0.0036   0.7123  -0.1329     10     43
366        0.0022   0.6900  -0.1173     10     37
386        0.0075   0.6385  -0.3279     10     30
392       -0.0074   0.7482  -0.1465     10     35
401       -0.0020   0.6759  -0.1392     10     45
------------------------------------------------


## Step 11: Summary Statistics

In [34]:
# Compile results into DataFrame for easier viewing
results_df = pd.DataFrame([
    {
        'target': target_idx,
        'r_rest': validation_results[target_idx]['r_rest'],
        'r_stim': validation_results[target_idx]['r_stim'],
        'r_delta': validation_results[target_idx]['r_delta'],
        'n_hcp': validation_results[target_idx]['n_hcp'],
        'n_tms': validation_results[target_idx]['n_tms']
    }
    for target_idx in sorted(validation_results.keys())
])

# Summary statistics
print("\n=== SUMMARY STATISTICS ===")
print(f"\nNumber of targets validated: {len(results_df)}")
print(f"\nREST FC Correlation (r_rest):")
print(f"  Mean: {results_df['r_rest'].mean():.4f}")
print(f"  Std:  {results_df['r_rest'].std():.4f}")
print(f"  Min:  {results_df['r_rest'].min():.4f}")
print(f"  Max:  {results_df['r_rest'].max():.4f}")

print(f"\nSTIM FC Correlation (r_stim):")
print(f"  Mean: {results_df['r_stim'].mean():.4f}")
print(f"  Std:  {results_df['r_stim'].std():.4f}")
print(f"  Min:  {results_df['r_stim'].min():.4f}")
print(f"  Max:  {results_df['r_stim'].max():.4f}")

print(f"\nDelta FC Correlation (r_delta):")
print(f"  Mean: {results_df['r_delta'].mean():.4f}")
print(f"  Std:  {results_df['r_delta'].std():.4f}")
print(f"  Min:  {results_df['r_delta'].min():.4f}")
print(f"  Max:  {results_df['r_delta'].max():.4f}")

print(f"\n=== INTERPRETATION ===")
r_delta_mean = results_df['r_delta'].mean()
r_stim_mean = results_df['r_stim'].mean()
print(f"Delta > Stim: {r_delta_mean > r_stim_mean}")
if r_delta_mean > r_stim_mean:
    print("  ✓ Model captures stimulus-induced FC changes (delta > stim)")
else:
    print("  ✗ Model does not capture stimulus-induced FC specificity (delta ≤ stim)")

print(f"\nCross-dataset generalization: Mean r_stim = {r_stim_mean:.4f}")
if r_stim_mean > 0.3:
    print("  ✓ Good generalization (r > 0.3)")
elif r_stim_mean > 0.1:
    print("  ~ Moderate generalization (0.1 < r < 0.3)")
else:
    print("  ✗ Poor generalization (r < 0.1)")


=== SUMMARY STATISTICS ===

Number of targets validated: 10

REST FC Correlation (r_rest):
  Mean: -0.0000
  Std:  0.0045
  Min:  -0.0074
  Max:  0.0075

STIM FC Correlation (r_stim):
  Mean: 0.6975
  Std:  0.0308
  Min:  0.6385
  Max:  0.7482

Delta FC Correlation (r_delta):
  Mean: -0.1589
  Std:  0.0895
  Min:  -0.3279
  Max:  -0.0567

=== INTERPRETATION ===
Delta > Stim: False
  ✗ Model does not capture stimulus-induced FC specificity (delta ≤ stim)

Cross-dataset generalization: Mean r_stim = 0.6975
  ✓ Good generalization (r > 0.3)


## Results Table

In [35]:
# Display full results table
print("\nDetailed Results by Target:")
print(results_df.to_string(index=False))

# Save results
out_dir = os.path.join(tms_preproc_dir, "HCP_to_TMS_validation")
os.makedirs(out_dir, exist_ok=True)

results_pkl = os.path.join(out_dir, "validation_results.pkl")
with open(results_pkl, 'wb') as f:
    pickle.dump({
        'results_df': results_df,
        'validation_results': validation_results,
        'empirical_by_target': dict(empirical_by_target),
        'simulated_by_target': dict(simulated_by_target)
    }, f)
print(f"\n✓ Results saved to: {results_pkl}")


Detailed Results by Target:
 target    r_rest   r_stim   r_delta  n_hcp  n_tms
    155 -0.002096 0.707684 -0.056715     10     42
    220  0.004041 0.725646 -0.290366     10     28
    231 -0.000031 0.670633 -0.204846     10     44
    305  0.004131 0.696931 -0.076309     10     43
    342 -0.003217 0.708760 -0.096638     10     41
    359 -0.003563 0.712292 -0.132867     10     43
    366  0.002179 0.690025 -0.117298     10     37
    386  0.007457 0.638507 -0.327943     10     30
    392 -0.007372 0.748153 -0.146490     10     35
    401 -0.001997 0.675925 -0.139206     10     45

✓ Results saved to: /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/preprocessed_subjects_tms_fmri/HCP_to_TMS_validation/validation_results.pkl
